# Paso 1: Instalar dependencias
Ejecutá esto en una celda para instalar los conectores necesarios:

In [ ]:
!pip install -q pyspark sqlalchemy sqlalchemy-pytds

# Paso 2: Crear la conexión y cargar tablas a PySpark
Usamos SQLAlchemy con pytds, y luego lo convertimos a DataFrame de Spark:

In [ ]:
import os
from sqlalchemy import create_engine
from pyspark.sql import SparkSession

# Crear sesión de Spark
spark = SparkSession.builder.getOrCreate()

# Datos de conexión
server = os.environ.get('DW_SERVER', 'SQLSERVER_CATEDRA')
database = 'DW'
username = os.environ.get('DW_USER', 'sa')
password = os.environ['DW_PASSWORD']  # credencial saneada: ver README

# Conexión usando pytds (This part is not used for Spark JDBC connection)
# engine = create_engine(f"mssql+pytds://{username}:{password}@{server}/{database}")

# Función para cargar una tabla como Spark DataFrame
def cargar_tabla(nombre_tabla):
    # Modified JDBC URL to include trustServerCertificate=true to disable SSL verification (use with caution)
    jdbc_url = f"jdbc:sqlserver://{server};databaseName={database};user={username};password={password};encrypt=true;trustServerCertificate=true;"
    jdbc_driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

    df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", nombre_tabla) \
        .option("user", username) \
        .option("password", password) \
        .option("driver", jdbc_driver) \
        .load()
    return df

# Paso 3: Descargar el driver JDBC y configurarlo en Colab
El conector JDBC no está por defecto en Colab, necesitás descargarlo:

In [ ]:
# Descargar y mover el driver JDBC para SQL Server
!wget -q https://repo1.maven.org/maven2/com/microsoft/sqlserver/mssql-jdbc/11.2.0.jre8/mssql-jdbc-11.2.0.jre8.jar -O mssql-jdbc.jar

# Configurar el driver en la sesión de Spark
spark = SparkSession.builder \
    .config("spark.jars", "mssql-jdbc.jar") \
    .getOrCreate()

# Paso 4: Cargar las tablas necesarias

In [ ]:
ft_contratos = cargar_tabla("FT_Contratos")
ft_inventario_contenedores = cargar_tabla("FT_Inventario_Contenedores")
dim_tipo_contenedor = cargar_tabla("DIM_Tipo_Contenedor")
dim_tipo_envio = cargar_tabla("DIM_Tipo_Envio")
dim_tiempo = cargar_tabla("DIM_Tiempo")
dim_puerto = cargar_tabla("DIM_Puerto")

# Paso 5: Confirmar que está todo OK

In [ ]:
ft_contratos.show(5)
ft_contratos.printSchema()

+---------+--------+----------+-------------+---------+-----------------+------------+---------------+-------+---------+--------------+
|Id_Tiempo|Id_Viaje|Id_Cliente|Id_Mercaderia|Id_Puerto|Id_EstadoContrato|Id_TipoEnvio|Id_TipoContrato|Cant_kg|Cant_teus|Cant_contratos|
+---------+--------+----------+-------------+---------+-----------------+------------+---------------+-------+---------+--------------+
| 20230101|       1|         1|            1|        1|                1|           1|              1|  39000|        1|             1|
| 20230101|       1|         1|            2|        1|                1|           1|              1|  38000|        1|             1|
| 20230101|       5|         1|            1|        1|                1|           1|              1|  39000|        1|             1|
| 20230101|       5|         1|            2|        1|                1|           1|              1|  38000|        1|             1|
| 20230101|       9|         1|            1|   

# Consulta 1: Puertos que mas han participado como origen o destino de contratos


In [ ]:
from pyspark.sql.functions import col, count, max as spark_max

# Fecha límite (10 años)
fecha_limite = 2015

# Unir con DIM_Tiempo para filtrar los últimos 10 años
contratos_recientes = ft_contratos.join(dim_tiempo, "Id_Tiempo") \
    .filter(col("Anio") >= fecha_limite)

# Contar ocurrencias por Id_Puerto
puerto_contratos = contratos_recientes.groupBy("Id_Puerto") \
    .agg(count("*").alias("Total_contratos"))

# Obtener el/los máximos
max_puertos = puerto_contratos.select(spark_max("Total_contratos").alias("maximo"))
puertos_top = puerto_contratos.join(max_puertos, puerto_contratos["Total_contratos"] == max_puertos["maximo"])

# Agregar nombres de puerto
puertos_top.join(dim_puerto, "Id_Puerto").select("Nombre", "Total_contratos").show()

+--------------------+---------------+
|              Nombre|Total_contratos|
+--------------------+---------------+
|Puerto de Buenos ...|             84|
+--------------------+---------------+



# Consulta 2: Tipos de Contenedor que han tenido más de 500 transportes esta semana

In [ ]:
from pyspark.sql.functions import lit, col, count, max as spark_max

# Semana actual (ejemplo: 1 al 30 de junio 2025)
fecha_actual = 2025
mes_actual = 6
dias_semana = list(range(1, 30))

# Filtrar por semana actual
contenedores_semana = ft_inventario_contenedores.join(dim_tiempo, "Id_Tiempo") \
    .filter((col("Anio") == fecha_actual) & (col("Mes") == mes_actual) & (col("Dia").isin(dias_semana)))

# Agrupar y filtrar
conteo_contenedores = contenedores_semana.groupBy("Id_TipoContenedor") \
    .agg(count("*").alias("Total_transportes")) \
    .filter(col("Total_transportes") > 500)

# Agregar descripción
conteo_contenedores.join(dim_tipo_contenedor, "Id_TipoContenedor").select("nombreContenedor", "Total_transportes").show()

+----------------+-----------------+
|nombreContenedor|Total_transportes|
+----------------+-----------------+
+----------------+-----------------+



# Consulta 3: Tipos de envíos contratados en todos los meses del año


In [ ]:
from pyspark.sql.functions import countDistinct

# Unir con tiempo
envios_por_mes = ft_contratos.join(dim_tiempo, "Id_Tiempo")

# Contar cuántos meses únicos tiene cada tipo de envío
envios_mensuales = envios_por_mes.groupBy("Id_TipoEnvio") \
    .agg(countDistinct("Mes").alias("Meses_contratados"))

# Filtrar los que estén en los 12 meses
envios_todo_el_anio = envios_mensuales.filter(col("Meses_contratados") > 6)

# Agregar descripción
envios_todo_el_anio.join(dim_tipo_envio, "Id_TipoEnvio").select("Descripcion").show()

+-----------+
|Descripcion|
+-----------+
|       Door|
+-----------+



# Consulta 4: Estación más fructífera (por TEUS o KG)

In [ ]:
from pyspark.sql.functions import when, sum as spark_sum

# Unir con tiempo
contratos_con_tiempo = ft_contratos.join(dim_tiempo, "Id_Tiempo")

# Asignar estación
contratos_con_estacion = contratos_con_tiempo.withColumn("Estacion", when(col("Mes").isin(12, 1, 2), "Verano")
                                                         .when(col("Mes").isin(3, 4, 5), "Otoño")
                                                         .when(col("Mes").isin(6, 7, 8), "Invierno")
                                                         .when(col("Mes").isin(9, 10, 11), "Primavera"))

# Sumar TEUS por estación (o cambiar por Cant_kg si preferís)
resultado = contratos_con_estacion.groupBy("Estacion") \
    .agg(spark_sum("Cant_teus").alias("Total_teus"))

# Ordenar por mayor valor
resultado.orderBy(col("Total_teus").desc()).show()

+---------+----------+
| Estacion|Total_teus|
+---------+----------+
| Invierno|       224|
|   Verano|       180|
|    Otoño|        59|
|Primavera|        59|
+---------+----------+

